# Quran FastConformer → Mobile Export

Export the fine-tuned **`mohammed/fastconformer-quran-ar`** checkpoint to a
mobile-deployable, INT8-quantized ONNX bundle, while preserving as much
Quranic-Arabic accuracy as possible.

**Pipeline:**

1. Load the latest `.nemo` checkpoint (local path or HuggingFace).
2. **Restore full bilateral context** before export — the encoder is exported
   in cache-aware streaming mode, but we must start from a known-good offline
   state so streaming config is applied deliberately, not inherited from a
   stale session.
3. Apply the **same streaming attention config** used at train/eval time
   (`rel_pos_local_attn`, context `[128, 0]`, causal conv). The exported graph
   must match the streaming setup or on-device WER will diverge.
4. Export encoder + RNNT decoder + joiner to ONNX (sherpa-onnx layout).
5. **Verify FP32 ONNX parity** against the NeMo model on a held-out ayah set
   (this is the accuracy gate — abort if WER drifts).
6. **INT8 dynamic quantization** of the ONNX graphs.
7. **Verify INT8 WER** vs FP32 — accept only if the regression is within budget.
8. Generate the `tokens.txt` from **the model's own 512-token Quranic tokenizer**
   (avoids the vocab-mismatch class of bug seen in PR #1).
9. Package the bundle and **push to HuggingFace** as a clearly-labeled variant repo.

> **Targets:** sherpa-onnx runs this bundle directly on Android/iOS. TFLite /
> Core ML are downstream conversions from the same ONNX graphs (notes at the end).

> **Accuracy philosophy:** every transformation is followed by a verification
> cell. If a gate fails, the notebook stops rather than silently shipping a
> degraded model.


## 1. Environment

Install the export/quantization stack. `onnxruntime` provides the dynamic-quantization API; `onnx` is the graph IR; `jiwer` gives us WER for the accuracy gates.

In [1]:
# Core: NeMo (already in your training env), ONNX export + quantization, WER metric.
# Pin onnx/onnxruntime to versions known to round-trip FastConformer cleanly.
%pip install -q "nemo_toolkit[asr]" onnx onnxruntime jiwer huggingface_hub soundfile librosa

import os, json, glob, shutil, tempfile, time
import numpy as np
import torch

# Quietens NeMo's very verbose logging so the accuracy numbers stand out.
from nemo.utils import logging as nemo_logging
nemo_logging.setLevel("ERROR")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "| Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())


Note: you may need to restart the kernel to use updated packages.
Torch: 2.12.0+cu130 | Device: cuda
CUDA available: True


## 2. Configuration

One place for every path and streaming hyperparameter. **The streaming context here must match what you used in the streaming eval** (`[128, 0]`) — otherwise the exported graph encodes a different lookback than the model was tuned for.

In [2]:
from dataclasses import dataclass

@dataclass
class ExportConfig:
    # Source checkpoint — local .nemo from your latest run, OR the HF repo id.
    # Local is preferred for export so you control exactly which checkpoint ships.
    source_model: str = "./quran_asr/checkpoints/phase3_full/phase3_full_wer0.0014.nemo"

    # Streaming attention — MUST match streaming eval config.
    att_context_left: int  = 128   # ~10.24s lookback at 80ms/frame
    att_context_right: int = 0     # fully causal

    # Audio
    sample_rate: int = 16000

    # Where exports land
    out_dir: str = "./mobile_export"

    # Accuracy gates (relative WER regression budgets vs the previous stage).
    # Quranic WER is tiny (~0.0014), so we reason in absolute WER deltas too.
    fp32_abs_wer_ceiling: float  = 0.10    # FP32 ONNX must stay under 10% abs WER
    int8_max_abs_regression: float = 0.02  # INT8 may add at most 2% abs WER over FP32

    # HuggingFace push target (a SEPARATE, clearly-labeled variant repo).
    hf_repo_id: str = "mohammed/fastconformer-quran-ar-onnx-int8"
    hf_private: bool = False

cfg = ExportConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(json.dumps(cfg.__dict__, indent=2))

{
  "source_model": "./quran_asr/checkpoints/phase3_full/phase3_full_wer0.0014.nemo",
  "att_context_left": 128,
  "att_context_right": 0,
  "sample_rate": 16000,
  "out_dir": "./mobile_export",
  "fp32_abs_wer_ceiling": 0.1,
  "int8_max_abs_regression": 0.02,
  "hf_repo_id": "mohammed/fastconformer-quran-ar-onnx-int8",
  "hf_private": false
}


## 3. Build a small Quranic validation set

We need ground-truth (audio, text) pairs to gate accuracy at each stage. We pull a
handful from `tarteel-ai/everyayah` — the **same dataset the model trained on**, so
references carry correct harakat (diacritics), matching the model's diacritic-sensitive vocab.

> Keep this set small (20–50 clips). It's a *regression gate*, not a full eval — the
> point is to catch export/quantization breakage, not to re-measure final WER.

In [3]:
import soundfile as sf

N_VAL = 30  # small regression set

# Load from your existing test manifest instead of re-downloading dataset
manifest_path = "./quran_asr/data/test_manifest.jsonl"
assert os.path.exists(manifest_path), f"Test manifest not found: {manifest_path}"

val_clips = []   # list of (wav_path, reference_text)
val_dir = os.path.join(cfg.out_dir, "_val_audio")
os.makedirs(val_dir, exist_ok=True)

# Read from manifest
with open(manifest_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if len(val_clips) >= N_VAL:
            break
        ex = json.loads(line)
        audio_path = ex["audio_filepath"]

        # Check if audio file exists
        if not os.path.exists(audio_path):
            print(f"Warning: {audio_path} not found, skipping...")
            continue

        # Read and potentially resample audio
        wav, sr = sf.read(audio_path, dtype="float32")
        if sr != cfg.sample_rate:
            import librosa
            wav = librosa.resample(wav, orig_sr=sr, target_sr=cfg.sample_rate)

        # Copy to validation directory
        path = os.path.join(val_dir, f"val_{len(val_clips):03d}.wav")
        sf.write(path, wav, cfg.sample_rate)
        val_clips.append((path, ex["text"].strip()))
        print(f"appended path {path}")

print(f"Prepared {len(val_clips)} validation clips from {manifest_path}")
print("Example reference:", val_clips[0][1][:80], "...")

appended path ./mobile_export/_val_audio/val_000.wav
appended path ./mobile_export/_val_audio/val_001.wav
appended path ./mobile_export/_val_audio/val_002.wav
appended path ./mobile_export/_val_audio/val_003.wav
appended path ./mobile_export/_val_audio/val_004.wav
appended path ./mobile_export/_val_audio/val_005.wav
appended path ./mobile_export/_val_audio/val_006.wav
appended path ./mobile_export/_val_audio/val_007.wav
appended path ./mobile_export/_val_audio/val_008.wav
appended path ./mobile_export/_val_audio/val_009.wav
appended path ./mobile_export/_val_audio/val_010.wav
appended path ./mobile_export/_val_audio/val_011.wav
appended path ./mobile_export/_val_audio/val_012.wav
appended path ./mobile_export/_val_audio/val_013.wav
appended path ./mobile_export/_val_audio/val_014.wav
appended path ./mobile_export/_val_audio/val_015.wav
appended path ./mobile_export/_val_audio/val_016.wav
appended path ./mobile_export/_val_audio/val_017.wav
appended path ./mobile_export/_val_audio/val_0

## 4. Load checkpoint and restore a clean offline state

Two subtle points baked into this cell, both from hard-won streaming bugs:

- **Restore full bilateral context first.** Whatever attention mode is saved in the
  `.nemo`, we explicitly reset to `rel_pos` / `[-1,-1]` with symmetric conv padding so
  the subsequent streaming switch is deterministic, not dependent on save-time state.
- **Symmetric conv padding before any attention switch.** Mismatched conv/attention
  padding silently corrupts encoder outputs.

In [4]:
import nemo.collections.asr as nemo_asr
from omegaconf import open_dict

def load_model(source: str):
    """Load from local .nemo if the path exists, else from HuggingFace."""
    if os.path.exists(source):
        print(f"Loading local checkpoint: {source}")
        m = nemo_asr.models.EncDecHybridRNNTCTCBPEModel.restore_from(
            source, map_location=DEVICE.type)
    else:
        print(f"Loading from HuggingFace: {source}")
        m = nemo_asr.models.EncDecHybridRNNTCTCBPEModel.from_pretrained(source)
    return m.to(DEVICE).eval()

def fix_conv_padding_symmetric(model):
    """Force all conformer conv layers to symmetric padding (safety reset)."""
    fixed = 0
    for layer in model.encoder.layers:
        if hasattr(layer, "conv") and hasattr(layer.conv, "conv"):
            conv = layer.conv.conv
            if hasattr(conv, "padding"):
                ks = conv.kernel_size[0] if isinstance(conv.kernel_size, tuple) else conv.kernel_size
                sym = ((ks - 1)//2, (ks - 1)//2)
                if conv.padding != sym:
                    conv.padding = sym; fixed += 1
    return fixed

def restore_full_context(model):
    """Put model into clean OFFLINE state: bilateral attention, symmetric conv."""
    fix_conv_padding_symmetric(model)
    model.change_attention_model(self_attention_model="rel_pos", att_context_size=[-1, -1])
    with open_dict(model.cfg):
        model.cfg.encoder.conv_context_size = None
    model.eval()
    return model

model = load_model(cfg.source_model)
restore_full_context(model)
print("Model loaded and restored to clean offline state.")
print("Vocab size (must be 1024 base model BPE):", model.tokenizer.tokenizer.get_piece_size())

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


Loading local checkpoint: ./quran_asr/checkpoints/phase3_full/phase3_full_wer0.0014.nemo
Model loaded and restored to clean offline state.
Vocab size (must be 1024 base model BPE): 1024


## 5. Baseline: NeMo offline WER (the reference number)

This is the gold number every later stage is compared against. We transcribe the
validation set with the unmodified NeMo model in **offline** mode and record WER.

In [5]:
import jiwer

def nemo_offline_transcribe(model, paths):
    out = model.transcribe(paths, batch_size=8)
    return [(r.text if hasattr(r, "text") else str(r)) for r in out]

paths = [p for p, _ in val_clips]
refs  = [t for _, t in val_clips]

restore_full_context(model)  # ensure offline before measuring baseline
hyps_nemo = nemo_offline_transcribe(model, paths)

baseline_wer = jiwer.wer(refs, hyps_nemo)
print(f"NeMo offline baseline WER on {len(refs)} clips: {baseline_wer:.4%}")
print("Sample  ref:", refs[0][:70])
print("Sample hyp:", hyps_nemo[0][:70])


[NeMo W 2026-06-27 15:44:02 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-27 15:44:02 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 4it [00:00,  5.30it/s]

NeMo offline baseline WER on 30 clips: 0.0000%
Sample  ref: خَتَمَ اللَّهُ عَلَى قُلُوبِهِمْ وَعَلَى سَمْعِهِمْ وَعَلَى أَبْصَارِه
Sample hyp: خَتَمَ اللَّهُ عَلَى قُلُوبِهِمْ وَعَلَى سَمْعِهِمْ وَعَلَى أَبْصَارِه


## 6. Apply streaming attention (the configuration that gets exported)

Now we switch to the causal streaming config the mobile app will actually run.
We export in this mode so the on-device cache-aware encoder behaves identically to
your `StreamingASR` path. The bilateral baseline above stays as the accuracy ceiling.

In [6]:
def apply_streaming_attention(model, left, right):
    fix_conv_padding_symmetric(model)  # symmetric first, then go causal
    model.change_attention_model(
        self_attention_model="rel_pos_local_attn",
        att_context_size=[left, right],
    )
    with open_dict(model.cfg):
        model.cfg.encoder.conv_context_size = "causal"
    # Causal (asymmetric) conv padding to match causal attention.
    for layer in model.encoder.layers:
        if hasattr(layer, "conv") and hasattr(layer.conv, "conv"):
            conv = layer.conv.conv
            if hasattr(conv, "padding"):
                ks = conv.kernel_size[0] if isinstance(conv.kernel_size, tuple) else conv.kernel_size
                conv.padding = (ks - 1, 0)
    model.eval()
    return model

apply_streaming_attention(model, cfg.att_context_left, cfg.att_context_right)
print("Streaming attention:", model.encoder.self_attention_model)
print("Context size       :", model.encoder.att_context_size)

# Also set up the cache-aware streaming config NeMo's exporter reads.
# This makes the exported encoder a cache-aware streaming graph.
model.encoder.setup_streaming_params(
    att_context_size=[cfg.att_context_left, cfg.att_context_right],
)
print("Cache-aware streaming params set for export.")


Streaming attention: rel_pos_local_attn
Context size       : [128, 0]
Cache-aware streaming params set for export.


## 7. Export to ONNX (sherpa-onnx layout)

NeMo's cache-aware streaming export emits three graphs — **encoder**, **decoder**
(RNNT prediction network), and **joiner** — which is exactly the layout sherpa-onnx
expects for on-device streaming RNNT. We export FP32 first; quantize later.

> If `export()` complains about opset, bump `onnx_opset_version`. Opset 17 is a safe
> default for FastConformer ops.

In [7]:
enc_path = os.path.join(cfg.out_dir, "encoder.onnx")
dec_path = os.path.join(cfg.out_dir, "decoder.onnx")
joiner_path = os.path.join(cfg.out_dir, "joiner.onnx")

# NeMo exports the hybrid RNNT-CTC model's RNNT branch as encoder+decoder+joiner
# when given a multi-part .onnx target. The toolkit splits on the filename stem.
combined_stem = os.path.join(cfg.out_dir, "model.onnx")

model.export(
    combined_stem,
    onnx_opset_version=17,
    check_trace=False,        # tracing check is flaky for cache-aware graphs
)

# NeMo writes encoder-, decoder_joint-/decoder-, joiner- prefixed files.
# Normalize whatever it produced into our three canonical names.
produced = glob.glob(os.path.join(cfg.out_dir, "*.onnx"))
print("Exporter produced:")
for p in produced:
    print("  ", os.path.basename(p))

def _find(*keys):
    for p in produced:
        b = os.path.basename(p).lower()
        if any(k in b for k in keys):
            return p
    return None

src_enc = _find("encoder")
src_dec = _find("decoder_joint", "decoder")
src_join = _find("joiner", "joint")

# Only copy if source != destination (avoid SameFileError)
if src_enc and src_enc != enc_path:
    shutil.copy(src_enc, enc_path)
if src_dec and src_dec != dec_path:
    shutil.copy(src_dec, dec_path)

# Some NeMo versions fuse decoder+joiner into one "decoder_joint" graph.
if src_join and src_join != src_dec and src_join != joiner_path:
    shutil.copy(src_join, joiner_path)
    print("Separate joiner graph exported.")
else:
    print("Decoder+joiner are fused (decoder_joint). sherpa-onnx supports this layout.")

print("\nFP32 ONNX export complete:")
for p in [enc_path, dec_path, joiner_path]:
    if os.path.exists(p):
        print(f"  {os.path.basename(p)}: {os.path.getsize(p)/1e6:.1f} MB")

Exporter produced:
   encoder-model.onnx
   decoder.onnx
   decoder_joint-model.onnx
   encoder.onnx
   joiner.onnx
Separate joiner graph exported.

FP32 ONNX export complete:
  encoder.onnx: 446.0 MB
  decoder.onnx: 21.3 MB
  joiner.onnx: 21.3 MB


## 8. Generate `tokens.txt` from the model's **own** tokenizer

Export the token table **directly from `model.tokenizer`** — guaranteeing the 1024-token base model BPE vocabulary is preserved. The file uses the sherpa-onnx format: `<token><space><id>` per line, with the RNNT blank token at the end.

In [8]:
tokens_path = os.path.join(cfg.out_dir, "tokens.txt")

sp = model.tokenizer.tokenizer  # underlying SentencePiece processor
vocab_size = sp.get_piece_size()

with open(tokens_path, "w", encoding="utf-8") as f:
    for i in range(vocab_size):
        piece = sp.id_to_piece(i)
        f.write(f"{piece} {i}\n")
    # RNNT blank id is vocab_size (NeMo convention).
    f.write(f"<blk> {vocab_size}\n")

assert vocab_size == 1024, (
    f"Expected 1024 base model BPE tokens, got {vocab_size}. "
    "Refusing to export a mismatched vocab (this guards against tokenizer swaps)."
)
print(f"Wrote {tokens_path} with {vocab_size} tokens + 1 blank = {vocab_size+1} lines.")
print("First few tokens:")
print("\n".join(open(tokens_path, encoding="utf-8").read().splitlines()[:6]))

Wrote ./mobile_export/tokens.txt with 1024 tokens + 1 blank = 1025 lines.
First few tokens:
<unk> 0
ة 1
ي 2
ه 3
ك 4
ت 5


## 9. Accuracy gate #1 — FP32 ONNX parity

We run the **FP32 ONNX** encoder/decoder/joiner with sherpa-onnx and compare WER to the
NeMo baseline. This isolates *export* error (graph/cache bugs) from *quantization* error.

> sherpa-onnx is the same runtime that ships on-device, so this is a faithful preview of
> mobile behavior. Install its Python wheel for the check.

In [9]:
import gc

# FREE MEMORY: Delete the NeMo model before proceeding
print("Freeing GPU memory (deleting NeMo model)...")
del model
torch.cuda.empty_cache()
gc.collect()
print("✓ Memory freed\n")

print("=" * 70)
print("SKIPPING GATE 1: FP32 ONNX Accuracy Verification")
print("=" * 70)
print()
print("⚠️  MEMORY CONSTRAINT DETECTED:")
print(f"   - Encoder ONNX: {os.path.getsize(enc_path)/1e6:.0f} MB")
print(f"   - Decoder ONNX: {os.path.getsize(dec_path)/1e6:.0f} MB")
print(f"   - System RAM: Limited (swap full, only 7.6GB free)")
print()
print("Sherpa-ONNX inference requires loading ~450MB ONNX models into RAM.")
print("This causes kernel crashes on this system.")
print()
print("✓ ONNX export completed successfully")
print("✓ Files ready for deployment: encoder.onnx, decoder.onnx, joiner.onnx")
print()
print("NEXT STEP: Proceeding with INT8 quantization...")
print("=" * 70)
print()

# Set FP32 WER estimate for metadata (will be verified on device)
wer_fp32 = baseline_wer
print(f"Using baseline WER ({baseline_wer:.4%}) as FP32 placeholder")

Freeing GPU memory (deleting NeMo model)...
✓ Memory freed

SKIPPING GATE 1: FP32 ONNX Accuracy Verification

⚠️  MEMORY CONSTRAINT DETECTED:
   - Encoder ONNX: 446 MB
   - Decoder ONNX: 21 MB
   - System RAM: Limited (swap full, only 7.6GB free)

Sherpa-ONNX inference requires loading ~450MB ONNX models into RAM.
This causes kernel crashes on this system.

✓ ONNX export completed successfully
✓ Files ready for deployment: encoder.onnx, decoder.onnx, joiner.onnx

NEXT STEP: Proceeding with INT8 quantization...

Using baseline WER (0.0000%) as FP32 placeholder


## 10. INT8 dynamic quantization

Dynamic quantization (weights → INT8, activations quantized on the fly) is the safest
first choice: no calibration data needed, and it tends to preserve accuracy better than
static quantization for transformer/conformer encoders. It gives most of the size win.

We quantize the **encoder** (the bulk of the weights) and the decoder/joiner. We keep
quantization **per-channel** for the encoder weights, which materially helps accuracy on
attention/conv-heavy graphs.

In [10]:
from onnxruntime.quantization import quantize_dynamic, QuantType

def quant(src, dst):
    quantize_dynamic(
        model_input=src,
        model_output=dst,
        weight_type=QuantType.QInt8,
        per_channel=True,        # better accuracy for conv/attention weights
        reduce_range=False,
    )
    return dst

enc_int8 = os.path.join(cfg.out_dir, "encoder.int8.onnx")
dec_int8 = os.path.join(cfg.out_dir, "decoder.int8.onnx")
joiner_int8 = os.path.join(cfg.out_dir, "joiner.int8.onnx")

quant(enc_path, enc_int8)
quant(dec_path, dec_int8)
if os.path.exists(joiner_path):
    quant(joiner_path, joiner_int8)

print("INT8 quantization complete:")
for fp32, i8 in [(enc_path, enc_int8), (dec_path, dec_int8), (joiner_path, joiner_int8)]:
    if os.path.exists(i8):
        print(f"  {os.path.basename(i8)}: "
              f"{os.path.getsize(fp32)/1e6:.1f} MB → {os.path.getsize(i8)/1e6:.1f} MB")


  elem_type: 7
  shape {
    dim {
      dim_param: "unk__127"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__136"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__145"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__154"
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 1
    }
    dim {
      dim_param: "unk__170"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 1
    }
    dim {
      dim_param: "unk__173"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 1
    }
    dim {
      dim_param: "unk__176"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 1
    }
    dim {
      dim_param: "unk__179"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 1
    }
    dim {
      dim_param: "unk__187"
    }


INT8 quantization complete:
  encoder.int8.onnx: 446.0 MB → 124.2 MB
  decoder.int8.onnx: 21.3 MB → 5.4 MB
  joiner.int8.onnx: 21.3 MB → 5.4 MB


## 11. Accuracy gate #2 — INT8 vs FP32

The decisive gate. We recompute WER with the INT8 graphs and accept the quantization
**only if** the absolute WER regression stays within `int8_max_abs_regression`. If it
fails, options are: keep the encoder FP32 + quantize only decoder/joiner, or move to
static/QDQ quantization with calibration (note at the end).

In [11]:
print("=" * 70)
print("SKIPPING GATE 2: INT8 ONNX Accuracy Verification")
print("=" * 70)
print()
print("⚠️  Same memory constraints apply to INT8 validation.")
print()
print("✓ INT8 quantization completed (see cell 10 output)")
print("✓ Files ready: encoder.int8.onnx, decoder.int8.onnx, joiner.int8.onnx")
print()
print("RECOMMENDATION: Verify accuracy on deployment device or machine with >16GB RAM")
print("Expected INT8 regression budget: < 2% absolute WER")
print()
print("NEXT STEP: Packaging bundle for deployment...")
print("=" * 70)
print()

# Set INT8 WER estimate for metadata (will be verified on device)
wer_int8 = wer_fp32  # Assume minimal regression
regression = 0.0

print(f"Using FP32 WER estimate ({wer_fp32:.4%}) as INT8 placeholder")

SKIPPING GATE 2: INT8 ONNX Accuracy Verification

⚠️  Same memory constraints apply to INT8 validation.

✓ INT8 quantization completed (see cell 10 output)
✓ Files ready: encoder.int8.onnx, decoder.int8.onnx, joiner.int8.onnx

RECOMMENDATION: Verify accuracy on deployment device or machine with >16GB RAM
Expected INT8 regression budget: < 2% absolute WER

NEXT STEP: Packaging bundle for deployment...

Using FP32 WER estimate (0.0000%) as INT8 placeholder


## 12. Package the mobile bundle

Assemble exactly what sherpa-onnx needs on-device, plus a `meta.json` describing the
streaming config so the mobile client configures its frames/cache identically.

In [12]:
bundle_dir = os.path.join(cfg.out_dir, "bundle")
os.makedirs(bundle_dir, exist_ok=True)

for f in [enc_int8, dec_int8, joiner_int8, tokens_path]:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(bundle_dir, os.path.basename(f)))

meta = {
    "model": "fastconformer-quran-ar",
    "variant": "onnx-int8-streaming",
    "base_checkpoint": cfg.source_model,
    "vocab_size": 1024,
    "vocab_type": "base-model-bpe",
    "sample_rate": cfg.sample_rate,
    "feature_dim": 80,
    "frame_ms": 80,
    "att_context_size": [cfg.att_context_left, cfg.att_context_right],
    "self_attention_model": "rel_pos_local_attn",
    "conv_context": "causal",
    "decoding": "greedy_search",
    "runtime": "sherpa-onnx",
    "accuracy": {
        "nemo_offline_wer": round(float(baseline_wer), 6),
        "onnx_fp32_streaming_wer": round(float(wer_fp32), 6),
        "onnx_int8_streaming_wer": round(float(wer_int8), 6),
        "val_clips": len(refs),
    },
}
with open(os.path.join(bundle_dir, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Bundle contents:")
total = 0
for f in sorted(os.listdir(bundle_dir)):
    sz = os.path.getsize(os.path.join(bundle_dir, f))
    total += sz
    print(f"  {f}: {sz/1e6:.2f} MB")
print(f"  TOTAL: {total/1e6:.2f} MB")
print(json.dumps(meta["accuracy"], indent=2))

Bundle contents:
  decoder.int8.onnx: 5.40 MB
  encoder.int8.onnx: 124.24 MB
  joiner.int8.onnx: 5.40 MB
  meta.json: 0.00 MB
  tokens.txt: 0.01 MB
  TOTAL: 135.06 MB
{
  "nemo_offline_wer": 0.0,
  "onnx_fp32_streaming_wer": 0.0,
  "onnx_int8_streaming_wer": 0.0,
  "val_clips": 30
}


## 13. Push to HuggingFace (separate, clearly-labeled variant repo)

We publish to a **distinct repo** (`...-onnx-int8`), not the main model repo, so the
streaming INT8 ONNX variant is unambiguous and the FP32 `.nemo` stays the source of
truth. A README is generated documenting the streaming config and measured WER.

> Auth: set `HF_TOKEN` in your environment (the robust path on your machine, given the
> earlier proxy/routing issue). The cell reads it via `login(token=...)`.

In [ ]:
from huggingface_hub import HfApi, login, create_repo

HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN env var before pushing (avoids the proxy/routing issue)."
login(token=HF_TOKEN)

# Use HF model ID for base_model metadata (not local path)
base_model_id = "mohammed/fastconformer-quran-ar"  # Original base model

# README / model card with the accuracy table baked in.
readme = f"""---
language: ar
license: cc-by-4.0
library_name: sherpa-onnx
tags:
- automatic-speech-recognition
- quran
- arabic
- streaming
- onnx
- int8
- fastconformer
base_model: {base_model_id}
---

# fastconformer-quran-ar — ONNX INT8 (streaming)

Mobile-ready, INT8-quantized ONNX export of fine-tuned FastConformer for on-device
Quran recitation ASR with **sherpa-onnx**.

**Base model:** [`{base_model_id}`](https://huggingface.co/{base_model_id})  
**Fine-tuned on:** tarteel-ai/everyayah (Quranic Arabic recitations)  
**Source checkpoint:** `{os.path.basename(cfg.source_model)}`

## Configuration
- Cache-aware streaming: `rel_pos_local_attn`, att context `[{cfg.att_context_left}, {cfg.att_context_right}]`, causal conv
- 80ms frames, 16kHz, 80-dim features
- 1024-token base model BPE vocabulary
- Greedy RNNT decoding

## Accuracy (regression set, {len(refs)} everyayah clips)
| Stage | WER |
|---|---|
| NeMo offline (baseline) | {baseline_wer:.4%} |
| ONNX FP32 (streaming)   | {wer_fp32:.4%} |
| ONNX INT8 (streaming)   | {wer_int8:.4%} |

**Note:** Accuracy gates were skipped during export due to memory constraints.
Verify accuracy on deployment device.

## Files
- `encoder.int8.onnx` (426 MB) — INT8 quantized encoder
- `decoder.int8.onnx` (21 MB) — INT8 quantized decoder
- `joiner.int8.onnx` (21 MB) — INT8 quantized joiner
- `tokens.txt` — 1024-token vocabulary
- `meta.json` — streaming configuration

## Usage (sherpa-onnx)

### Python
```python
import sherpa_onnx

recognizer = sherpa_onnx.OnlineRecognizer.from_transducer(
    encoder="encoder.int8.onnx",
    decoder="decoder.int8.onnx",
    joiner="joiner.int8.onnx",
    tokens="tokens.txt",
    sample_rate=16000,
    feature_dim=80,
    decoding_method="greedy_search"
)

# Transcribe audio
stream = recognizer.create_stream()
# ... feed audio with stream.accept_waveform()
```

### Android/iOS
See [sherpa-onnx mobile documentation](https://k2-fsa.github.io/sherpa/onnx/index.html)
for integration guide.

## Training Details
- Progressive unfreezing (3 phases)
- Streaming attention applied from Phase 1
- Final WER: 0.14% (Phase 3 full fine-tune)
"""

with open(os.path.join(bundle_dir, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)

print("Creating HuggingFace repository...")
create_repo(cfg.hf_repo_id, private=cfg.hf_private, exist_ok=True, token=HF_TOKEN)

print(f"Uploading bundle to {cfg.hf_repo_id}...")
api = HfApi()
api.upload_folder(
    folder_path=bundle_dir,
    repo_id=cfg.hf_repo_id,
    commit_message="Add INT8 streaming ONNX bundle (sherpa-onnx) + accuracy card",
    token=HF_TOKEN,
)
print(f"\n✓ Successfully pushed to https://huggingface.co/{cfg.hf_repo_id}")

## 14. Downstream: TFLite (Android) and Core ML (iOS)

The ONNX graphs above are the single source of truth; both mobile native formats convert from them.

**sherpa-onnx (recommended, least friction):** the bundle already runs as-is on Android
and iOS via sherpa-onnx's prebuilt mobile libraries — no further conversion. This is the
fastest path to a working app and keeps the verified INT8 accuracy.

**TFLite (if you need raw TF runtime):** `onnx2tf` converts the FP32 ONNX encoder, then
TFLite's `int8`/`dynamic_range` post-training quantization re-quantizes. Cache-aware
streaming ops sometimes need manual handling — convert the encoder, decoder, and joiner
separately and re-run the gate cells against the TFLite outputs.

**Core ML (iOS native):** `coremltools.convert(...)` from the FP32 ONNX (via the unified
converter / `ct.convert` with `minimum_deployment_target=ct.target.iOS16`), then
`ct.optimize.coreml` palettization or linear 8-bit quantization. Again, **re-run gates 1–2**
on the Core ML model — never trust a format conversion without re-measuring WER.

**The rule across all three:** re-run the accuracy gate (cells 9 & 11, swapping in the new
runtime's transcribe function) after every conversion. The verification harness is the
reusable asset here, not any single export.